In [171]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

########################################
YEAR = 2024 # <-- CHANGE YEAR
DOC_A = "sbm_2024"    # <-- FIRST DOCUMENT
DOC_B = "tkh_2024"    # <-- SECOND DOCUMENT
########################################

BASE = Path(".")
CLEAN_FOLDER = BASE / "data_clean" / f"mdna_clean_{YEAR}"
OUTPUT_FOLDER = BASE / "outputs" / f"pairwise_{YEAR}"

# Only create OUTPUT folder (data_clean must already exist)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

file_a = CLEAN_FOLDER / f"{DOC_A}.txt"
file_b = CLEAN_FOLDER / f"{DOC_B}.txt"

# Strict existence check (no dummy files)
if not file_a.exists():
    raise FileNotFoundError(f"Required file not found: {file_a.resolve()}")

if not file_b.exists():
    raise FileNotFoundError(f"Required file not found: {file_b.resolve()}")

# Read the files
text_a = file_a.read_text(encoding="utf-8", errors="ignore")
text_b = file_b.read_text(encoding="utf-8", errors="ignore")

documents = [text_a, text_b]

# =========================
# TF-IDF → cosine similarity
# =========================
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)
sim_matrix = cosine_similarity(tfidf_matrix)

df_sim = pd.DataFrame(
    sim_matrix,
    index=[DOC_A, DOC_B],
    columns=[DOC_A, DOC_B]
)

# =========================
# Average similarity (excl. diagonal)
# =========================
vals = df_sim.values.astype(float)
np.fill_diagonal(vals, np.nan)
avg_similarity = np.nanmean(vals)

pairwise_similarity = df_sim.loc[DOC_A, DOC_B]

print("\nSimilarity matrix:")
print(df_sim)
print(f"\nPairwise cosine similarity {DOC_A} vs {DOC_B}: {pairwise_similarity:.6f}")
print(f"Average cosine similarity (excluding diagonal): {avg_similarity:.6f}")

# =========================
# Save outputs
# =========================
name_tag = f"{DOC_A}__{DOC_B}"

df_sim.to_csv(OUTPUT_FOLDER / f"{name_tag}_cosine_similarity.csv")

pd.DataFrame([{
    "year": YEAR,
    "doc_a": DOC_A,
    "doc_b": DOC_B,
    "pairwise_cosine_similarity": pairwise_similarity,
    "avg_cosine_similarity_excl_diag": avg_similarity
}]).to_csv(OUTPUT_FOLDER / f"{name_tag}_summary.csv", index=False)

print(f"\nSaved outputs to: {OUTPUT_FOLDER.resolve()}")



Similarity matrix:
          sbm_2024  tkh_2024
sbm_2024  1.000000  0.184498
tkh_2024  0.184498  1.000000

Pairwise cosine similarity sbm_2024 vs tkh_2024: 0.184498
Average cosine similarity (excluding diagonal): 0.184498

Saved outputs to: /Users/pnlmf036/Documents/Thesis_Textanalysis/outputs/pairwise_2024
